In [1]:
"""
Build assumption_master sheet: lsdb_features
-----
Extracts LSDB-specific feature tables in ONE long-format sheet with a
`feature` discriminator column. Consolidates:

- waiting_periods    (Basic + AIDS)
- instalments        (Basic + AIDS)
- reduction_period   (period → factor + income tapering flag)
- backdating         (Basic + AIDS)
- db_definitions     (Any Occ/Own/Bedridden × Basic/AIDS)
- nra_match          (Red Period × NRA → table number)

Schema: feature | cost_basis | key | value | notes
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "lsdb_features"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")

Building: lsdb_features


In [2]:
def find_all_cells_any_col(ws, label, max_row=1200, max_col=30):
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)

def peek(ws, row, col, span=5):
    parts = []
    for dc in range(0, span + 1):
        v = ws.cell(row=row, column=col + dc).value
        parts.append(f"{get_col_letter(col + dc)}{row}={v!r}")
    return " | ".join(parts)

# Locate every LSDB feature section anchor
DIAGNOSTIC_LABELS = [
    "Lumpsum disability definitions",
    "Any Occupation",
    "Own or Similar",
    "Own Occupation",
    "Bedridden",
    "Waiting Periods",
    "Basic WP table",
    "Instalments",
    "Basic Installment Factors",
    "Annual for 5 years",
    "Reduction period",
    "Income Tapering",
    "Basic Backdating Factors",
    "LSDB to GLA AIDS Factor",
    "Red Period _ NRA",
    "Table to use",
    "NRA",
    "NRA to use",
]

print("=" * 90)
print("DIAGNOSTIC — LSDB feature anchor locations")
print("=" * 90)

for label in DIAGNOSTIC_LABELS:
    hits = find_all_cells_any_col(ws, label)
    # Filter to LSDB block (rows 250-313)
    lsdb_hits = [(r, c) for r, c in hits if 250 <= r <= 313]
    other_hits = [(r, c) for r, c in hits if not (250 <= r <= 313)]
    print(f"\n'{label}': {len(lsdb_hits)} match(es) in LSDB block")
    for i, (r, c) in enumerate(lsdb_hits):
        print(f"  [{i}] {peek(ws, r, c)}")
    if other_hits:
        print(f"  (also {len(other_hits)} match(es) outside LSDB block)")

DIAGNOSTIC — LSDB feature anchor locations

'Lumpsum disability definitions': 1 match(es) in LSDB block
  [0] B271='Lumpsum disability definitions' | C271='Basic DB Definitions' | D271='AIDS DB Definitions' | E271=None | F271=None | G271=None

'Any Occupation': 1 match(es) in LSDB block
  [0] B272='Any Occupation' | C272=-0.25 | D272=-0.25 | E272=None | F272=None | G272=None

'Own or Similar': 1 match(es) in LSDB block
  [0] B273='Own or Similar' | C273=0 | D273=0 | E273=None | F273=None | G273=None

'Own Occupation': 1 match(es) in LSDB block
  [0] B274='Own Occupation' | C274=0.3 | D274=0.3 | E274=None | F274=None | G274=None

'Bedridden': 1 match(es) in LSDB block
  [0] B275='Bedridden' | C275=-0.85 | D275=-0.85 | E275=None | F275=None | G275=None

'Waiting Periods': 1 match(es) in LSDB block
  [0] B278='Waiting Periods' | C278='Basic WP table' | D278='AIDS WP table' | E278=None | F278=None | G278=None

'Basic WP table': 1 match(es) in LSDB block
  [0] C278='Basic WP table' | D278='

In [3]:
# Preview rows around each sub-table anchor
print("--- DB Definitions (rows 271-275, cols B-D) ---")
for row in range(271, 276):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row}: B={b!r:30s} C={c!r:10s} D={d!r}")

print("\n--- Waiting Periods (rows 278-284, cols B-D) ---")
for row in range(278, 285):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row}: B={b!r:20s} C={c!r:10s} D={d!r}")

print("\n--- Instalments (rows 286-293, cols B-D) ---")
for row in range(286, 294):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row}: B={b!r:25s} C={c!r:10s} D={d!r}")

print("\n--- Reduction Period (rows 295-302, cols B-D) ---")
for row in range(295, 303):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row}: B={b!r:20s} C={c!r:10s} D={d!r}")

print("\n--- Backdating (rows 304-310, cols B-D) ---")
for row in range(304, 311):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row}: B={b!r:35s} C={c!r:10s} D={d!r}")

print("\n--- Red Period × NRA grid (rows 255-275, cols I-M) ---")
for row in range(255, 276):
    i = ws.cell(row=row, column=9).value   # I
    j = ws.cell(row=row, column=10).value  # J
    k = ws.cell(row=row, column=11).value  # K
    l = ws.cell(row=row, column=12).value  # L
    m = ws.cell(row=row, column=13).value  # M
    print(f"  Row {row}: I={i!r:20s} J={j!r:10s} K={k!r:5s} L={l!r:5s} M={m!r}")

--- DB Definitions (rows 271-275, cols B-D) ---
  Row 271: B='Lumpsum disability definitions' C='Basic DB Definitions' D='AIDS DB Definitions'
  Row 272: B='Any Occupation'               C=-0.25      D=-0.25
  Row 273: B='Own or Similar'               C=0          D=0
  Row 274: B='Own Occupation'               C=0.3        D=0.3
  Row 275: B='Bedridden'                    C=-0.85      D=-0.85

--- Waiting Periods (rows 278-284, cols B-D) ---
  Row 278: B='Waiting Periods'    C='Basic WP table' D='AIDS WP table'
  Row 279: B=6                    C=0          D=0
  Row 280: B=12                   C=-0.125     D=-0.125
  Row 281: B=18                   C=-0.2       D=-0.2
  Row 282: B=24                   C=-0.3       D=-0.3
  Row 283: B=0                    C=0.025      D=0.025
  Row 284: B=None                 C=None       D=None

--- Instalments (rows 286-293, cols B-D) ---
  Row 286: B='Instalments'             C='Basic Installment Factors' D='AIDS Installment Factors'
  Row 287: B=4

In [4]:
all_rows: list[dict] = []

def add_row(feature, cost_basis, key, value, source_row, notes=""):
    all_rows.append({
        "feature": feature,
        "cost_basis": cost_basis,
        "key": key,
        "value": value,
        "source_row": source_row,
        "notes": notes,
    })

# =============================================================================
# 1. DB Definitions (rows 272-275, cols B/C/D)
#    Basic in col C, AIDS in col D
# =============================================================================
DB_DEFS = [
    (272, "any_occupation"),
    (273, "own_or_similar"),
    (274, "own_occupation"),
    (275, "bedridden"),
]
for row, key in DB_DEFS:
    basic = ws.cell(row=row, column=3).value  # C
    aids  = ws.cell(row=row, column=4).value  # D
    label = ws.cell(row=row, column=2).value  # B
    add_row("db_definitions", "basic", key, basic, row,
            f"Disability definition '{label}' - Basic")
    add_row("db_definitions", "aids", key, aids, row,
            f"Disability definition '{label}' - AIDS")

# =============================================================================
# 2. Waiting Periods (rows 279-283, cols B/C/D)
#    Key is number of months (or 0 as sentinel). Basic in C, AIDS in D.
# =============================================================================
WP_ROWS = [279, 280, 281, 282, 283]
for row in WP_ROWS:
    months = ws.cell(row=row, column=2).value  # B: 6, 12, 18, 24, 0
    basic  = ws.cell(row=row, column=3).value
    aids   = ws.cell(row=row, column=4).value
    key = f"months_{months}"
    add_row("waiting_periods", "basic", key, basic, row,
            f"Waiting period {months} months - Basic")
    add_row("waiting_periods", "aids", key, aids, row,
            f"Waiting period {months} months - AIDS")

# =============================================================================
# 3. Instalments (rows 287-292, cols B/C/D)
#    Keys: 40, 60, 80, 100, 120, "Annual for 5 years"
# =============================================================================
INSTAL_ROWS = [287, 288, 289, 290, 291, 292]
for row in INSTAL_ROWS:
    period = ws.cell(row=row, column=2).value  # B
    basic  = ws.cell(row=row, column=3).value
    aids   = ws.cell(row=row, column=4).value
    # Normalize key: numbers stay, strings snake_case
    if isinstance(period, str):
        key = period.lower().replace(" ", "_")
    else:
        key = str(int(period)) if isinstance(period, (int, float)) else str(period)
    add_row("instalments", "basic", key, basic, row,
            f"Instalment factor {period!r} - Basic")
    add_row("instalments", "aids", key, aids, row,
            f"Instalment factor {period!r} - AIDS")

# =============================================================================
# 4. Reduction Period (rows 296-301, cols B/C/D)
#    Two numeric cols: C=code, D=factor
#    Row 301 special: label "Income Tapering", C=6, D=0
# =============================================================================
REDUC_ROWS = [
    (296, "0"),
    (297, "36"),
    (298, "60"),
    (299, "96"),
    (300, "120"),
    (301, "income_tapering"),
]
for row, key in REDUC_ROWS:
    code   = ws.cell(row=row, column=3).value  # C
    factor = ws.cell(row=row, column=4).value  # D
    add_row("reduction_period", "code",   key, code,   row,
            f"Reduction period {key} - code")
    add_row("reduction_period", "factor", key, factor, row,
            f"Reduction period {key} - factor")

# =============================================================================
# 5. Backdating factors (rows 305-307, cols B/C)
#    3 rows, all zeros. No visible key → positional index.
#    Basic in col B, AIDS in col C.
# =============================================================================
BACKDATE_ROWS = [305, 306, 307]
for i, row in enumerate(BACKDATE_ROWS, start=1):
    basic = ws.cell(row=row, column=2).value  # B
    aids  = ws.cell(row=row, column=3).value  # C
    key = f"idx_{i}"
    add_row("backdating", "basic", key, basic, row,
            f"Backdating factor position {i} - Basic (source has no explicit key)")
    add_row("backdating", "aids", key, aids, row,
            f"Backdating factor position {i} - AIDS (source has no explicit key)")

# =============================================================================
# 6. LSDB to GLA AIDS Factor (row 309, single scalar)
# =============================================================================
lsdb_aids_factor = ws.cell(row=309, column=3).value  # C309 = 0.6
add_row("lsdb_to_gla_aids", "scalar", "value", lsdb_aids_factor, 309,
        "Scalar factor: LSDB AIDS rate as fraction of GLA AIDS rate")

# =============================================================================
# 7. Red Period × NRA → Table number (rows 256-273, cols I/J)
# =============================================================================
NRA_TABLE_ROWS = list(range(256, 274))  # 18 rows
for row in NRA_TABLE_ROWS:
    red_period_nra = ws.cell(row=row, column=9).value  # I
    table_num      = ws.cell(row=row, column=10).value # J
    if red_period_nra is None:
        continue
    add_row("red_period_to_table", "table_num", str(red_period_nra), table_num, row,
            f"Red Period × NRA lookup: {red_period_nra} → table {table_num}")

# =============================================================================
# 8. NRA → NRA to use (rows 256-275, cols L/M)
# =============================================================================
NRA_USE_ROWS = list(range(256, 276))  # 20 rows
for row in NRA_USE_ROWS:
    actual_nra = ws.cell(row=row, column=12).value  # L
    nra_to_use = ws.cell(row=row, column=13).value  # M
    if actual_nra is None:
        continue
    key = str(int(actual_nra)) if isinstance(actual_nra, (int, float)) else str(actual_nra)
    add_row("nra_to_use", "nra_to_use", key, nra_to_use, row,
            f"Actual NRA {actual_nra} → NRA to use {nra_to_use}")

# --- Summary ---
from collections import Counter
feature_counts = Counter(r["feature"] for r in all_rows)
print(f"Total rows harvested: {len(all_rows)}")
print(f"\nRows per feature:")
for feat, cnt in sorted(feature_counts.items()):
    print(f"  {feat:22s}: {cnt}")

print(f"\nFirst 8 rows preview:")
for r in all_rows[:8]:
    print(f"  {r['feature']:22s} | {r['cost_basis']:10s} | {r['key']:20s} = "
          f"{r['value']} ({type(r['value']).__name__})")

Total rows harvested: 87

Rows per feature:
  backdating            : 6
  db_definitions        : 8
  instalments           : 12
  lsdb_to_gla_aids      : 1
  nra_to_use            : 20
  red_period_to_table   : 18
  reduction_period      : 12
  waiting_periods       : 10

First 8 rows preview:
  db_definitions         | basic      | any_occupation       = -0.25 (float)
  db_definitions         | aids       | any_occupation       = -0.25 (float)
  db_definitions         | basic      | own_or_similar       = 0 (int)
  db_definitions         | aids       | own_or_similar       = 0 (int)
  db_definitions         | basic      | own_occupation       = 0.3 (float)
  db_definitions         | aids       | own_occupation       = 0.3 (float)
  db_definitions         | basic      | bedridden            = -0.85 (float)
  db_definitions         | aids       | bedridden            = -0.85 (float)


In [5]:
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

for r in all_rows:
    v = r["value"]
    r["param_type"] = classify(v)
    r["value_str"] = "" if v is None else str(v)

df = pl.DataFrame(all_rows).select([
    "feature",
    "cost_basis",
    "key",
    pl.col("value_str").alias("value"),
    "param_type",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
key_cols = ["feature", "cost_basis", "key"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

expected_features = {
    "db_definitions", "waiting_periods", "instalments",
    "reduction_period", "backdating", "lsdb_to_gla_aids",
    "red_period_to_table", "nra_to_use",
}
actual_features = set(df["feature"].unique().to_list())
missing = expected_features - actual_features
unexpected = actual_features - expected_features
assert not missing, f"Missing features: {missing}"
assert not unexpected, f"Unexpected features: {unexpected}"

# =============================================================================
# Frozen value expectations
# =============================================================================
def get_val(feature, cost_basis, key):
    row = df.filter(
        (pl.col("feature") == feature)
        & (pl.col("cost_basis") == cost_basis)
        & (pl.col("key") == key)
    )
    assert row.height == 1, f"Missing: {feature}/{cost_basis}/{key}"
    return float(row["value"][0])

frozen = [
    # DB definitions
    ("db_definitions",   "basic",  "any_occupation",  -0.25),
    ("db_definitions",   "basic",  "own_or_similar",   0.0),
    ("db_definitions",   "aids",   "bedridden",       -0.85),
    # Waiting periods
    ("waiting_periods",  "basic",  "months_6",         0.0),
    ("waiting_periods",  "basic",  "months_24",       -0.3),
    ("waiting_periods",  "aids",   "months_0",         0.025),
    # Instalments
    ("instalments",      "basic",  "40",              -0.17),
    ("instalments",      "basic",  "120",             -0.45),
    ("instalments",      "aids",   "annual_for_5_years", -0.2),
    # Reduction period
    ("reduction_period", "code",   "0",                1),
    ("reduction_period", "factor", "120",              10),
    ("reduction_period", "code",   "income_tapering",  6),
    # Backdating (all zero)
    ("backdating",       "basic",  "idx_1",            0),
    ("backdating",       "aids",   "idx_3",            0),
    # Scalar
    ("lsdb_to_gla_aids", "scalar", "value",            0.6),
    # NRA lookups
    ("red_period_to_table", "table_num", "2_60",       11),
    ("red_period_to_table", "table_num", "6_65",       16),
    ("nra_to_use",       "nra_to_use", "50",           60),
    ("nra_to_use",       "nra_to_use", "69",           65),
]

for feature, cost_basis, key, expected in frozen:
    actual = get_val(feature, cost_basis, key)
    assert abs(actual - float(expected)) < 1e-6, (
        f"{feature}/{cost_basis}/{key}: got {actual}, expected {expected}"
    )

# Row-count expectations per feature
expected_counts = {
    "db_definitions":       8,   # 4 defs × 2 cost_basis
    "waiting_periods":     10,   # 5 periods × 2
    "instalments":         12,   # 6 periods × 2
    "reduction_period":    12,   # 6 periods × 2
    "backdating":           6,   # 3 rows × 2
    "lsdb_to_gla_aids":     1,   # scalar
    "red_period_to_table": 18,   # 18 codes
    "nra_to_use":          20,   # 20 NRAs
}
for feat, expected in expected_counts.items():
    actual = df.filter(pl.col("feature") == feat).height
    assert actual == expected, (
        f"Feature {feat}: got {actual} rows, expected {expected}"
    )

print(f"✓ {df.height} rows validated")
print(f"✓ {len(expected_features)} features present")
print(f"✓ {len(frozen)} frozen value checks passed")
print(f"✓ All 8 feature row-counts match expectations")

print(f"\nType distribution:")
print(df.group_by("param_type").len().sort("param_type"))

print(f"\nDataFrame (head 20):")
print(df.head(20))
print(f"\nDataFrame (tail 10):")
print(df.tail(10))

✓ 87 rows validated
✓ 8 features present
✓ 19 frozen value checks passed
✓ All 8 feature row-counts match expectations

Type distribution:
shape: (2, 2)
┌────────────┬─────┐
│ param_type ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ float      ┆ 27  │
│ int        ┆ 60  │
└────────────┴─────┘

DataFrame (head 20):
shape: (20, 7)
┌─────────────────┬────────────┬────────────────┬───────┬────────────┬────────────┬────────────────┐
│ feature         ┆ cost_basis ┆ key            ┆ value ┆ param_type ┆ source_row ┆ notes          │
│ ---             ┆ ---        ┆ ---            ┆ ---   ┆ ---        ┆ ---        ┆ ---            │
│ str             ┆ str        ┆ str            ┆ str   ┆ str        ┆ i64        ┆ str            │
╞═════════════════╪════════════╪════════════════╪═══════╪════════════╪════════════╪════════════════╡
│ db_definitions  ┆ basic      ┆ any_occupation ┆ -0.25 ┆ float      ┆ 272        ┆ Disability     │
│                 ┆            ┆    

In [6]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote lsdb_features.parquet
  Size:    4.22 KB
  SHA-256: b1e008aeaa7c237d...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [7]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "LSDB feature tables consolidated. 8 features in one long-format sheet: "
        "db_definitions (4×2), waiting_periods (5×2), instalments (6×2), "
        "reduction_period (6×2), backdating (3×2, all zeros), lsdb_to_gla_aids "
        "(scalar 0.6), red_period_to_table (18 codes), nra_to_use (20 NRAs). "
        "Backdating rows have no explicit key in source — positional idx_1/2/3 "
        "used with note flagging ambiguity."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated

BUILD COMPLETE: lsdb_features
